# 07_01 Bundesland-aware calendar: baseline data versus fixed data

Both fits use the **same feature code** and the same model configuration. The only
difference is the underlying transaction dataset:

- `data/processed/transactions` — built with a **Niedersachsen-only** public-holiday
  calendar applied to every store.
- `data/processed/transactions_fixed` — built with **per-Bundesland** calendars, output of
  the corrected `src/data/pipeline.py`.

## The defect

The store network spans two Bundeslaender — 132 stores in Niedersachsen (≈79.5% of volume)
and 57 in Nordrhein-Westfalen — whose public holidays differ. `is_active` was derived from
the NI calendar for all of them, which corrupts the data in **both** directions:

| Holiday | Valid in | What the old pipeline did to NW stores |
| --- | --- | --- |
| Fronleichnam, Allerheiligen | NW only | marked **active** while the stores were shut, so the model forecasts a normal trading day |
| Reformationstag | NI only | marked **inactive**, which **zeroes their real sales** |

The second is data destruction. Measured on 2025-10-31, the NW stores show `is_active` = 0%
and 0.0 kg against their own median of 54 kg per store-day.

Store Bundesland is assigned from the postal code, with explicit overrides for four border
municipalities (Salzbergen and Emsbueren carry 48xxx prefixes but lie in the Emsland;
Ibbenbueren carries 49xxx but lies in Kreis Steinfurt). The prefix rule agrees with observed
store closures on NW-only holidays for 97.9% of stores.

## What this comparison can and cannot show

The evaluation window (2026-03-02 to 2026-07-18) contains **exactly one** divergent
holiday — Fronleichnam on 2026-06-04. Reformationstag falls outside it. So the recovered
sales enter only through the **training history**, and the direct scoring effect is limited
to a single day. This comparison therefore under-tests the fix by construction, and that
should be kept in mind when reading the aggregate numbers.

Two views are reported:

1. **Own active rows** — each run scored on the rows it considers active. The fixed run
   legitimately benefits from no longer forecasting a closed day.
2. **Common rows** — the intersection, which isolates the effect of better *training* data
   from the effect of dropping bad evaluation rows.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (p.resolve() for p in [Path("../.."), Path(".")] if (p / "data").exists()), None
)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.data.preparation.distribute_sales_over_active_days import store_subdivisions

RESULTS = PROJECT_ROOT / "reports/results"
BASELINE = RESULTS / "run_baseline_data/forecasts/artikel_markt_multi7days_lightgbm_two_stage.csv"
FIXED = RESULTS / "run_fixed_data/forecasts/artikel_markt_multi7days_lightgbm_two_stage.csv"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})

COLUMNS = [
    "ARTIKEL_ID", "MARKT_ID", "origin", "period",
    "actual", "forecast", "is_active", "recent_occurrence_rate",
]

def load(path):
    chunks = [
        chunk[chunk.is_active == True].drop(columns="is_active")
        for chunk in pd.read_csv(path, usecols=COLUMNS, chunksize=2_000_000)
    ]
    frame = pd.concat(chunks, ignore_index=True).dropna(subset=["actual", "forecast"])
    frame["origin"] = pd.to_datetime(frame.origin)
    frame["period"] = pd.to_datetime(frame.period)
    return frame

baseline, fixed = load(BASELINE), load(FIXED)
subdivisions = store_subdivisions()

display(pd.DataFrame([{
    "baseline active rows": len(baseline),
    "fixed active rows": len(fixed),
    "row difference": len(fixed) - len(baseline),
    "baseline actual (kg)": baseline.actual.sum(),
    "fixed actual (kg)": fixed.actual.sum(),
}]).T.rename(columns={0: "Value"}).style.format("{:,.0f}"))

,Value
baseline active rows,"2,502,829"
fixed active rows,"2,498,967"
row difference,"-3,862"
baseline actual (kg),"2,661,458"
fixed actual (kg),"2,661,595"


## Verification: is the data actually fixed?

Straight from the two processed datasets, on the three dates where the calendars diverge.

In [2]:
import duckdb

con = duckdb.connect()
con.execute("PRAGMA threads=4")
con.register("store_state", subdivisions)

rows = []
for label, folder in [
    ("baseline", "transactions"), ("fixed", "transactions_fixed"),
]:
    glob = str(PROJECT_ROOT / f"data/processed/{folder}/*.parquet")
    frame = con.execute(f"""
        SELECT s.subdivision, t.DATE::DATE AS date,
               AVG(t.is_active::INT) AS active_share,
               SUM(t.ABVERKAUFTE_MENGE_KG) AS kg
        FROM read_parquet('{glob}') t
        JOIN store_state s USING (MARKT_ID)
        WHERE t.DATE::DATE IN (
            DATE '2025-06-19', DATE '2025-10-31', DATE '2026-06-04'
        )
        GROUP BY 1, 2
    """).fetchdf()
    frame["dataset"] = label
    rows.append(frame)

verification = pd.concat(rows, ignore_index=True).pivot_table(
    index=["date", "subdivision"], columns="dataset", values=["active_share", "kg"]
)
display(verification.style.format("{:,.2f}"))
display(Markdown(
    "*2025-06-19 and 2026-06-04 are Fronleichnam (NW only); 2025-10-31 is "
    "Reformationstag (NI only). In the fixed dataset the NW stores are closed on "
    "Fronleichnam and their Reformationstag sales are restored.*"
))

*2025-06-19 and 2026-06-04 are Fronleichnam (NW only); 2025-10-31 is Reformationstag (NI only). In the fixed dataset the NW stores are closed on Fronleichnam and their Reformationstag sales are restored.*

## Target metrics

In [3]:
def metrics(frame, column="forecast"):
    weekly = frame.groupby(["ARTIKEL_ID", "MARKT_ID", "origin"], observed=True).agg(
        a=("actual", "sum"), f=(column, "sum")
    )
    article_day = frame.groupby(["ARTIKEL_ID", "period"], observed=True).agg(
        a=("actual", "sum"), f=(column, "sum")
    )
    return {
        "row (article-store-day)": (frame[column] - frame.actual).abs().sum()
            / frame.actual.sum() * 100,
        "article-store-week": (weekly.f - weekly.a).abs().sum() / weekly.a.sum() * 100,
        "article-day": (article_day.f - article_day.a).abs().sum()
            / article_day.a.sum() * 100,
        "relative bias": (frame[column].sum() - frame.actual.sum())
            / frame.actual.sum() * 100,
    }

own = pd.DataFrame({
    "baseline data": metrics(baseline), "fixed data": metrics(fixed),
})
own["delta"] = own["fixed data"] - own["baseline data"]
display(own.style.set_caption("Each run on its own active rows").format("{:+.3f}", subset=["delta"]).format("{:.3f}", subset=["baseline data", "fixed data"]))

keys = ["ARTIKEL_ID", "MARKT_ID", "origin", "period"]
common = baseline.merge(
    fixed[keys + ["forecast"]].rename(columns={"forecast": "forecast_fixed"}),
    on=keys, how="inner", validate="one_to_one",
).merge(subdivisions, on="MARKT_ID", how="left")
common["subdivision"] = common.subdivision.fillna("NI")

shared = pd.DataFrame({
    "baseline data": metrics(common, "forecast"),
    "fixed data": metrics(common, "forecast_fixed"),
})
shared["delta"] = shared["fixed data"] - shared["baseline data"]
display(shared.style.set_caption(
    f"Common rows only ({len(common):,})"
).format("{:+.3f}", subset=["delta"]).format("{:.3f}", subset=["baseline data", "fixed data"]))

dropped = baseline.merge(fixed[keys], on=keys, how="left", indicator=True)
dropped = dropped[dropped._merge == "left_only"]
display(Markdown(
    f"**Rows the fix removed:** {len(dropped):,}, all on "
    f"{', '.join(str(d) for d in sorted(dropped.period.dt.date.unique()))}. "
    f"Their true demand is **{dropped.actual.sum():,.0f} kg** while the baseline model "
    f"forecast **{dropped.forecast.sum():,.0f} kg** for them — pure error against a day "
    "the stores were shut."
))

,baseline data,fixed data,delta
row (article-store-day),57.482,57.442,-0.040
article-store-week,36.938,37.124,+0.186
article-day,23.891,23.957,+0.067
relative bias,-0.605,-1.209,-0.604


,baseline data,fixed data,delta
row (article-store-day),57.420,57.438,+0.018
article-store-week,36.918,37.121,+0.203
article-day,23.847,23.957,+0.110
relative bias,-0.668,-1.211,-0.543


**Rows the fix removed:** 5,914, all on 2026-06-04. Their true demand is **0 kg** while the baseline model forecast **1,672 kg** for them — pure error against a day the stores were shut.

## Did the mechanism work where it should?

The fix should show up in two specific places: the NW stores on Fronleichnam itself, and
the pre-Fronleichnam stock-up day that the model previously could not see.

In [4]:
run_up = common[(common.period == "2026-06-03") & (common.subdivision == "NW")]
display(pd.DataFrame([{
    "day": "2026-06-03 (run-up to Fronleichnam, NW stores)",
    "rows": len(run_up),
    "actual (kg)": run_up.actual.sum(),
    "forecast/actual, baseline": run_up.forecast.sum() / run_up.actual.sum(),
    "forecast/actual, fixed": run_up.forecast_fixed.sum() / run_up.actual.sum(),
}]).T.rename(columns={0: "Value"}))

state_rows = []
for subdivision, group in common.groupby("subdivision"):
    before, after = metrics(group, "forecast"), metrics(group, "forecast_fixed")
    state_rows.append({
        "Bundesland": subdivision,
        "volume share": group.actual.sum() / common.actual.sum(),
        "row baseline": before["row (article-store-day)"],
        "row fixed": after["row (article-store-day)"],
        "row delta": after["row (article-store-day)"] - before["row (article-store-day)"],
        "bias baseline": before["relative bias"],
        "bias fixed": after["relative bias"],
    })
display(pd.DataFrame(state_rows).style.hide(axis="index").format({
    "volume share": "{:.1%}", "row baseline": "{:.3f}", "row fixed": "{:.3f}",
    "row delta": "{:+.3f}", "bias baseline": "{:+.2f}", "bias fixed": "{:+.2f}",
}))

origin_rows = []
for origin, group in common.groupby("origin"):
    origin_rows.append({
        "origin": origin.date(),
        "baseline": (group.forecast - group.actual).abs().sum() / group.actual.sum() * 100,
        "fixed": (group.forecast_fixed - group.actual).abs().sum() / group.actual.sum() * 100,
    })
origins = pd.DataFrame(origin_rows)
origins["delta"] = origins.fixed - origins.baseline
display(origins.style.hide(axis="index").format({
    "baseline": "{:.2f}", "fixed": "{:.2f}", "delta": "{:+.2f}",
}))
display(Markdown(
    f"**{int((origins.delta < 0).sum())} of {len(origins)} origins improved.** "
    "The Fronleichnam origin is 2026-06-01."
))

,Value
day,"2026-06-03 (run-up to Fronleichnam, NW stores)"
rows,5914
actual (kg),6337.9229
"forecast/actual, baseline",0.649948
"forecast/actual, fixed",0.908144


Bundesland,volume share,row baseline,row fixed,row delta,bias baseline,bias fixed
NI,79.9%,56.697,56.724,+0.027,-1.00,-1.68
NW,20.1%,60.296,60.277,-0.018,+0.64,+0.66


origin,baseline,fixed,delta
2026-03-02,55.03,55.62,+0.59
2026-03-09,61.34,60.92,-0.42
2026-03-16,57.03,57.48,+0.45
2026-03-23,62.26,62.05,-0.22
2026-03-30,49.02,48.70,-0.32
2026-04-06,58.51,58.53,+0.02
2026-04-13,65.73,67.63,+1.89
2026-04-20,58.33,58.43,+0.10
2026-04-27,55.21,54.61,-0.60
2026-05-04,56.91,56.67,-0.23


**11 of 20 origins improved.** The Fronleichnam origin is 2026-06-01.

## Conclusions

**The data is now correct, and the targeted mechanism works. The aggregate effect on this
evaluation window is flat, and relative bias moves the wrong way — but the window contains
only one divergent holiday, so it cannot settle the question either way.**

### The correctness fix is not in doubt

Read straight from the two datasets:

| Date | Holiday in | NW stores, baseline | NW stores, fixed |
| --- | --- | --- | --- |
| 2025-10-31 Reformationstag | NI only | inactive, **0 kg** | active, **9,721 kg restored** |
| 2025-06-19 Fronleichnam | NW only | active, 0 kg | correctly closed |
| 2026-06-04 Fronleichnam | NW only | active, 0 kg | correctly closed |

The baseline model forecast **1,672 kg on 2026-06-04 for stores that were shut**, against
0 kg of true demand — pure error, and only one of several such days per year. Across the
three-year history the Reformationstag defect alone erased roughly 29 tonnes of real sales
from the training data of 57 stores.

### The mechanism works exactly where it was predicted to

The pre-Fronleichnam stock-up day, which the model previously had no way to recognise for
NW stores:

**2026-06-03, NW stores: forecast/actual 0.650 → 0.908.** A 35% underforecast reduced to
9%. This is the clearest single confirmation in the batch, and it is precisely the
behaviour the state-aware `holiday_event_window` was meant to unlock.

### But the aggregate is flat, and bias degrades

| Grain | Baseline | Fixed | Delta |
| --- | ---: | ---: | ---: |
| row (own active rows) | 57.482 | 57.442 | **−0.040** |
| row (common rows) | 57.420 | 57.438 | +0.018 |
| article-store-week (common) | 36.918 | 37.121 | **+0.203** |
| article-day (common) | 23.847 | 23.957 | +0.110 |
| relative bias (common) | −0.668 | −1.211 | **−0.543** |

11 of 20 origins improved. The Fronleichnam origin (2026-06-01) improved by −0.27 pp; the
largest degradation, 2026-04-13 at +1.89 pp, is the post-Easter week and has nothing to do
with this change.

### Why the aggregate cannot be read as a verdict

Three reasons, in order of importance:

1. **The window under-tests the fix by construction.** It spans 2026-03-02 to 2026-07-18
   and contains exactly one divergent holiday. Reformationstag and Allerheiligen fall
   outside it, so the recovered sales enter only through training history. A window
   covering late October and November would test this properly.
2. **The bias movement is within this model's run-to-run variation.** The baseline fit here
   scores 57.482 / −0.605 where the published `06_02` run on the *same data* scored
   57.413 / −0.973. A feature-code change alone moved bias by +0.368 pp. Against that, a
   −0.543 pp delta cannot be confidently attributed to the data fix. The seed-repeat left
   open in `06_02` is now the blocking measurement, not an optional refinement.
3. **The state breakdown does not support a causal story.** NI carries 79.9% of volume, its
   underlying data barely changed, and yet its bias moved −1.00 → −1.68 while NW — the
   stores the fix actually touches — moved only +0.64 → +0.66 and improved slightly on row
   WAPE (−0.018). If the fix were driving the bias change, the sign pattern would be the
   other way round.

### Verdict

**Adopt the fix.** It is not a modelling choice to be judged on WAPE — it is a correctness
defect. The old pipeline told the model that shut stores were trading and that trading
stores were shut, and it deleted real sales. Keeping known-wrong data because a 20-week
window with one divergent holiday shows a flat metric would be the wrong trade.

What should **not** be claimed is a WAPE improvement. On this evidence the fix is
metric-neutral at row grain and slightly negative at the aggregated grains.

### Follow-ups

1. **Run the seed-repeat** (`06_02`, TODO 1) before drawing any further conclusions from
   sub-0.5 pp movements. Two consecutive batches have now produced deltas smaller than the
   apparent refit noise.
2. **Re-evaluate on a window containing Reformationstag or Allerheiligen** — moving
   `FIRST_ORIGIN` into the autumn would test the half of the defect this window cannot see.
3. **Carry the fix into the pooled feature tables.** Only the row-level calendar join is
   state-aware; the 15 history and cross-store aggregate joins still use the Niedersachsen
   calendar because they are pooled over stores and have no Bundesland key. That is a
   defensible approximation at 79.5% of volume, but it is an approximation.
4. **School holidays.** They are used nowhere in the model today, differ sharply between NI
   and NW, and are known months in advance — the same class of signal as this fix, and
   still untouched.

---

## P.S. — an article-specific event-position feature was tried and reverted

Once the state-aware calendar made it possible to measure event position against each
store's **own** Bundesland holidays, the residuals showed a clear pattern (five-seed
average, so well above the 0.053 pp noise floor established in `07_02`):

| Position | Forecast/actual |
| --- | ---: |
| 1–3 days before a holiday | **0.837 – 0.872** |
| 1 day after | 0.887 |
| 2 days after | **1.137** |

An oracle suggested this was worth a lot: de-biasing each position **pooled over articles**
gained nothing (−0.018 pp), while de-biasing **per article** gained **1.65 pp**. Article
responses at the same position are heterogeneous and opposite in sign (pre-holiday
forecast/actual p10 0.652 to p90 1.123), so a pooled correction cancels out — the same
diagnosis that made `06_02`'s article × temperature feature work.

So `event_position_expected_lift` was built on exactly that pattern: per-article,
per-position lifts fitted strictly pre-origin, weekday-normalised, empirical-Bayes shrunk
toward the pooled lift at the same position.

**It failed all three pre-registered predictions and was reverted.**

- Row WAPE 57.442 → 57.331. Against the five-seed mean of 57.397 rather than the single
  seed-42 baseline, that is −0.066 pp ≈ **1.2 sd — not separable from noise**. Weekly and
  article-day likewise.
- The new column ranked **below** the pooled `event_position_lift_*` columns it was meant
  to supersede (quantity stage: 24.4 versus 15.0) — the opposite of `06_02`, where the
  article-specific column jumped to rank 13.6.
- The pre-holiday positions it targeted got **worse** (+0.16 to +0.59 pp WAPE); the small
  aggregate movement came from post-holiday and ordinary days instead.
- Bias appeared to improve (−1.209 → −0.814) but −0.814 sits inside the no-feature seed
  range of −0.797 to −1.284. Seed 13 produced better bias with no feature at all.

**Why the oracle misled.** It fitted means over roughly 10–25 dates per article-position
cell, so it largely fitted noise; the 1.65 pp was mostly overfitting. By contrast the
temperature oracle in `05_10` fitted a *slope over ~900 daily observations* per article,
which is why it was reliable and was in fact delivered in full. **An oracle bound is only
as trustworthy as its cell counts** — that should be checked before quoting one.

The underlying effect is real and still unexploited. The estimable route is a coarser cut —
sourcing group × position, or occurrence band × position, where cells hold hundreds of
dates — keeping some heterogeneity while remaining measurable. The per-article version is
not.

Reverted in full: feature column, builder tables, contract entry, description and tests.
`FEATURE_BUILDER_VERSION` returned to `2026-08-07.4`.